<h1>Categorical data</h1>

<p>This is an introduction to pandas categorical data type, including a short comparison with R’s <code>factor</code>.</p>

<p><code>Categoricals</code> are a pandas data type corresponding to categorical variables in statistics. A categorical variable takes on a limited, and usually fixed, number of possible values (<code>categories</code>; <code>levels</code> in R). Examples are gender, social class, blood type, country affiliation, observation time or rating via Likert scales.</p>

<p>In contrast to statistical categorical variables, categorical data might have an order (e.g. ‘strongly agree’ vs ‘agree’ or ‘first observation’ vs. ‘second observation’), but numerical operations (additions, divisions, …) are not possible.</p>

<p>All values of categorical data are either in <code>categories</code> or <code>np.nan</code>. Order is defined by the order of <code>categories</code>, not lexical order of the values. Internally, the data structure consists of a <code>categories</code> array and an integer array of <code>codes</code> which point to the real value in
the <code>categories</code> array.</p>

<p>The categorical data type is useful in the following cases:</p>

<ul>

<li><p>A string variable consisting of only a few different values. Converting such a string variable to a categorical variable will save some memory, see <a href="https://pandas.pydata.org/docs/user_guide/categorical.html#categorical-memory">here</a>.</p></li>

<li><p>The lexical order of a variable is not the same as the logical order (“one”, “two”, “three”).
By converting to a categorical and specifying an order on the categories, sorting and min/max will use the logical order instead of the lexical order, see <a href="https://pandas.pydata.org/docs/user_guide/categorical.html#categorical-sort">here</a>.</p></li>

<li><p>As a signal to other Python libraries that this column should be treated as a categorical variable (e.g. to use suitable statistical methods or plot types).</p></li>

</ul>

<p>See also the <a href="https://pandas.pydata.org/docs/reference/arrays.html#api-arrays-categorical">API docs on categoricals</a>.</p>

# <h2>Object creation</h2>

## <h3>Series creation</h3>

<p>Categorical <code>Series</code> or columns in a <code>DataFrame</code> can be created in several ways:</p>

<p>By specifying <code>dtype="category"</code> when constructing a <code>Series</code>:</p>

In [270]:
import pandas as pd
import numpy as np

In [271]:
s = pd.Series(["a", "b", "c", "a"], dtype="category")

s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

<p>By converting an existing <code>Series</code> or column to a <code>category</code> dtype:</p>

In [272]:
df = pd.DataFrame({"A": ["a", "b", "c", "a"]})

df["B"] = df["A"].astype("category")

df

,A,B
0,a,a
1,b,b
2,c,c
3,a,a


<p>By using special functions, such as <a href="https://pandas.pydata.org/docs/reference/api/pandas.cut.html#pandas.cut" title="pandas.cut"><code>cut()</code></a>, which groups data into discrete bins. See the <a href="https://pandas.pydata.org/docs/user_guide/reshaping.html#reshaping-tile-cut">example on tiling</a> in the docs.</p>

In [273]:
df = pd.DataFrame({"value": np.random.randint(0, 100, 20)})

labels = ["{0} - {1}".format(i, i + 9) for i in range(0, 100, 10)]

df["group"] = pd.cut(x=df.value, bins=range(0, 105, 10), right=False, labels=labels)

df.head(10)

,value,group
0,32,30 - 39
1,75,70 - 79
2,81,80 - 89
3,28,20 - 29
4,38,30 - 39
5,29,20 - 29
6,98,90 - 99
7,73,70 - 79
8,90,90 - 99
9,87,80 - 89


<p>By passing a <a href="https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html#pandas.Categorical" title="pandas.Categorical"><code>pandas.Categorical</code></a> object to a <code>Series</code> or assigning it to a <code>DataFrame</code>.</p>

In [274]:
raw_cat = pd.Categorical(
    ["a", "b", "c", "a"], categories=["b", "c", "d"], ordered=False
)

s = pd.Series(raw_cat)

s

0    NaN
1      b
2      c
3    NaN
dtype: category
Categories (3, object): ['b', 'c', 'd']

In [275]:
df = pd.DataFrame({"A": ["a", "b", "c", "a"]})

df["B"] = raw_cat

df

,A,B
0,a,NaN
1,b,b
2,c,c
3,a,NaN


<p>Categorical data has a specific <code>category</code> <a href="https://pandas.pydata.org/docs/user_guide/basics.html#basics-dtypes">dtype</a>:</p>

In [276]:
df.dtypes

A      object
B    category
dtype: object

## <h3>DataFrame creation</h3>

<p>Similar to the previous section where a single column was converted to categorical, all columns in a <code>DataFrame</code> can be batch converted to categorical either during or after construction.</p>

<p>This can be done during construction by specifying <code>dtype="category"</code> in the <code>DataFrame</code> constructor:</p>

In [277]:
df = pd.DataFrame({"A": list("abca"), "B": list("bccd")}, dtype="category")

df.dtypes

A    category
B    category
dtype: object

<p>Note that the categories present in each column differ; the conversion is done column by column, so only labels present in a given column are categories:</p>

In [278]:
df["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (3, object): ['a', 'b', 'c']

In [279]:
df["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (3, object): ['b', 'c', 'd']

<p>Analogously, all columns in an existing <code>DataFrame</code> can be batch converted using <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html#pandas.DataFrame.astype" title="pandas.DataFrame.astype"><code>DataFrame.astype()</code></a>:</p>

In [280]:
df = pd.DataFrame({"A": list("abca"), "B": list("bccd")})

df_cat = df.astype("category")

df_cat.dtypes

A    category
B    category
dtype: object

<p>This conversion is likewise done column by column:</p>

In [281]:
df_cat["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (3, object): ['a', 'b', 'c']

In [282]:
df_cat["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (3, object): ['b', 'c', 'd']

## <h3>Controlling behavior</h3>

<p>In the examples above where we passed <code>dtype='category'</code>, we used the default behavior:</p>

<ol>
<li><p>Categories are inferred from the data.</p></li>
<li><p>Categories are unordered.</p></li>
</ol>

<p>To control those behaviors, instead of passing <code>'category'</code>, use an instance of <code>CategoricalDtype</code>.</p>

In [283]:
from pandas.api.types import CategoricalDtype

s = pd.Series(["a", "b", "c", "a"])

cat_type = CategoricalDtype(categories=["b", "c", "d"], ordered=True)

s_cat = s.astype(cat_type)

s_cat

0    NaN
1      b
2      c
3    NaN
dtype: category
Categories (3, object): ['b' < 'c' < 'd']

<p>Similarly, a <code>CategoricalDtype</code> can be used with a <code>DataFrame</code> to ensure that categories are consistent among all columns.</p>

In [284]:
from pandas.api.types import CategoricalDtype

df = pd.DataFrame({"A": list("abca"), "B": list("bccd")})

cat_type = CategoricalDtype(categories=list("abcd"), ordered=True)

df_cat = df.astype(cat_type)

df_cat["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (4, object): ['a' < 'b' < 'c' < 'd']

In [285]:
df_cat["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (4, object): ['a' < 'b' < 'c' < 'd']

<div class="alert alert-block alert-info">
<p>Note</p>
<p>To perform table-wise conversion, where all labels in the entire <code>DataFrame</code> are used as categories for each column, the <code>categories</code> parameter can be determined programmatically by <code>categories = pd.unique(df.to_numpy().ravel())</code>.</p>
</div>

<p>If you already have <code>codes</code> and <code>categories</code>, you can use the
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Categorical.from_codes.html#pandas.Categorical.from_codes" title="pandas.Categorical.from_codes"><code>from_codes()</code></a> constructor to save the factorize step during normal constructor mode:</p>

In [286]:
splitter = np.random.choice([0, 1], 5, p=[0.5, 0.5])

s = pd.Series(pd.Categorical.from_codes(splitter, categories=["train", "test"]))

## <h3>Regaining original data</h3>

<p>To get back to the original <code>Series</code> or NumPy array, use <code>Series.astype(original_dtype)</code> or <code>np.asarray(categorical)</code>:</p>

In [287]:
s = pd.Series(["a", "b", "c", "a"])

s

0    a
1    b
2    c
3    a
dtype: object

In [288]:
s2 = s.astype("category")

s2

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [289]:
s2.astype(str)

0    a
1    b
2    c
3    a
dtype: object

In [290]:
np.asarray(s2)

array(['a', 'b', 'c', 'a'], dtype=object)

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>In contrast to R’s <code>factor</code> function, categorical data is not converting input values to strings; categories will end up the same data type as the original values.</p>

</div>
<div class="admonition note">
<p class="admonition-title">Note</p>
<p>In contrast to R’s <code>factor</code> function, there is currently no way to assign/change labels at creation time. Use <code>categories</code> to change the categories after creation time.</p>
</div>

# <h2>CategoricalDtype</h2>

<p>A categorical’s type is fully described by</p>

<ol>
<li><p><code>categories</code>: a sequence of unique values and no missing values</p></li>
<li><p><code>ordered</code>: a boolean</p></li>
</ol>

<p>This information can be stored in a <code>CategoricalDtype</code>. The <code>categories</code> argument is optional, which implies that the actual categories should be inferred from whatever is present in the data when the <a href="https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html#pandas.Categorical" title="pandas.Categorical"><code>pandas.Categorical</code></a> is created.
The categories are assumed to be unordered by default.</p>

In [291]:
from pandas.api.types import CategoricalDtype

CategoricalDtype(["a", "b", "c"])

CategoricalDtype(categories=['a', 'b', 'c'], ordered=False, categories_dtype=object)

In [292]:
CategoricalDtype(["a", "b", "c"], ordered=True)

CategoricalDtype(categories=['a', 'b', 'c'], ordered=True, categories_dtype=object)

In [293]:
CategoricalDtype()

CategoricalDtype(categories=None, ordered=False, categories_dtype=None)

<p>A <code>CategoricalDtype</code> can be used in any place pandas expects a <code>dtype</code>. For example <a href="https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html#pandas.read_csv" title="pandas.read_csv"><code>pandas.read_csv()</code></a>,
<a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html#pandas.DataFrame.astype" title="pandas.DataFrame.astype"><code>pandas.DataFrame.astype()</code></a>, or in the <code>Series</code> constructor.</p>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>As a convenience, you can use the string <code>'category'</code> in place of a
<code>CategoricalDtype</code> when you want the default behavior of the categories being unordered, and equal to the set values present in the array. In other words, <code>dtype='category'</code> is equivalent to <code>dtype=CategoricalDtype()</code>.</p>
</div>

## <h3>Equality semantics</h3>

<p>Two instances of <code>CategoricalDtype</code> compare equal
whenever they have the same categories and order. When comparing two
unordered categoricals, the order of the <code>categories</code> is not considered.</p>

In [294]:
c1 = CategoricalDtype(["a", "b", "c"], ordered=False)

# Equal, since order is not considered when ordered=False
c1 == CategoricalDtype(["b", "c", "a"], ordered=False)

True

In [295]:
# Unequal, since the second CategoricalDtype is ordered
In [51]: c1 == CategoricalDtype(["a", "b", "c"], ordered=True)

False

<p>All instances of <code>CategoricalDtype</code> compare equal to the string <code>'category'</code>.</p>

In [296]:
c1 == "category"

True

# <h2>Description</h2>

<p>Using <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html#pandas.DataFrame.describe" title="pandas.DataFrame.describe"><code>describe()</code></a> on categorical data will produce similar
output to a <code>Series</code> or <code>DataFrame</code> of type <code>string</code>.</p>

In [297]:
cat = pd.Categorical(["a", "c", "c", np.nan], categories=["b", "a", "c"])

df = pd.DataFrame({"cat": cat, "s": ["a", "c", "c", np.nan]})

df.describe()

,cat,s
count,3,3
unique,2,2
top,c,c
freq,2,2


In [298]:
df["cat"].describe()

count     3
unique    2
top       c
freq      2
Name: cat, dtype: object

# <h2>Working with categories</h2>

<p>Categorical data has a <code>categories</code> and a <code>ordered</code> property, which list their possible values and whether the ordering matters or not. These properties are exposed as <code>s.cat.categories</code> and <code>s.cat.ordered</code>. If you don’t manually
specify categories and ordering, they are inferred from the passed arguments.</p>

In [299]:
s = pd.Series(["a", "b", "c", "a"], dtype="category")

s.cat.categories

Index(['a', 'b', 'c'], dtype='object')

In [300]:
s.cat.ordered

False

<p>It’s also possible to pass in the categories in a specific order:</p>

In [301]:
s = pd.Series(pd.Categorical(["a", "b", "c", "a"], categories=["c", "b", "a"]))

s.cat.categories

Index(['c', 'b', 'a'], dtype='object')

In [302]:
s.cat.ordered

False

<div class="alert alert-block alert-info">
<p>Note</p>
<p>New categorical data are <strong>not</strong> automatically ordered. You must explicitly pass <code>ordered=True</code> to indicate an ordered <code>Categorical</code>.</p>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>The result of <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html#pandas.Series.unique" title="pandas.Series.unique"><code>unique()</code></a> is not always the same as <code>Series.cat.categories</code>, because <code>Series.unique()</code> has a couple of guarantees, namely that it returns categories in the order of appearance, and it only includes values that are actually present.</p>

In [303]:
s = pd.Series(list("babc")).astype(CategoricalDtype(list("abcd")))

In [304]:
s

0    b
1    a
2    b
3    c
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [305]:
# categories
s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='object')

In [306]:
# uniques
s.unique()

['b', 'a', 'c']
Categories (4, object): ['a', 'b', 'c', 'd']

# <h3>Renaming categories</h3>

<p>Renaming categories is done by using the <code>rename_categories()</code> method:</p>

In [307]:
s = pd.Series(["a", "b", "c", "a"], dtype="category")

In [308]:
s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [309]:
new_categories = ["Group %s" % g for g in s.cat.categories]

s = s.cat.rename_categories(new_categories)

In [310]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (3, object): ['Group a', 'Group b', 'Group c']

In [311]:
# You can also pass a dict-like object to map the renaming
s = s.cat.rename_categories({1: "x", 2: "y", 3: "z"})

In [312]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (3, object): ['Group a', 'Group b', 'Group c']

<div class="alert alert-block alert-info">
<p>Note</p>
<p>In contrast to R’s <code>factor</code>, categorical data can have categories of other types than string.</p>
</div>

<p>Categories must be unique or a <code>ValueError</code> is raised:</p>

In [313]:
try:
    s = s.cat.rename_categories([1, 1, 1])
except ValueError as e:
    print("ValueError:", str(e))

ValueError: Categorical categories must be unique


<p>Categories must also not be <code>NaN</code> or a <code>ValueError</code> is raised:</p>

In [314]:
try:
    s = s.cat.rename_categories([1, 2, np.nan])
except ValueError as e:
    print("ValueError:", str(e))

ValueError: Categorical categories cannot be null


## <h3>Appending new categories</h3>

<p>Appending categories can be done by using the <code>add_categories()</code> method:</p>

In [315]:
s = s.cat.add_categories([4])

s.cat.categories

Index(['Group a', 'Group b', 'Group c', 4], dtype='object')

In [316]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (4, object): ['Group a', 'Group b', 'Group c', 4]

## <h3>Removing categories</h3>

<p>Removing categories can be done by using the <code>remove_categories()</code> method. Values which are removed are replaced by <code>np.nan</code>.:</p>

In [317]:
s = s.cat.remove_categories([4])

In [318]:
s

0    Group a
1    Group b
2    Group c
3    Group a
dtype: category
Categories (3, object): ['Group a', 'Group b', 'Group c']

## <h3>Removing unused categories</h3>

<p>Removing unused categories can also be done:</p>

In [319]:
s = pd.Series(pd.Categorical(["a", "b", "a"], categories=["a", "b", "c", "d"]))

In [320]:
s

0    a
1    b
2    a
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [321]:
s.cat.remove_unused_categories()

0    a
1    b
2    a
dtype: category
Categories (2, object): ['a', 'b']

## <h3>Setting categories</h3>

<p>If you want to do remove and add new categories in one step (which has some speed advantage), or simply set the categories to a predefined scale, use <code>set_categories()</code>.</p>

In [322]:
s = pd.Series(["one", "two", "four", "-"], dtype="category")

In [323]:
s

0     one
1     two
2    four
3       -
dtype: category
Categories (4, object): ['-', 'four', 'one', 'two']

In [324]:
s = s.cat.set_categories(["one", "two", "three", "four"])

In [325]:
s

0     one
1     two
2    four
3     NaN
dtype: category
Categories (4, object): ['one', 'two', 'three', 'four']

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>Be aware that <code>Categorical.set_categories()</code> cannot know whether some category is omitted
intentionally or because it is misspelled or (under Python3) due to a type difference (e.g.,
NumPy S1 dtype and Python strings). This can result in surprising behaviour!</p>
</div>
</section>
</section>

# <h2>Sorting and order</h2>

<p id="categorical-sort">If categorical data is ordered (<code>s.cat.ordered == True</code>), then the order of the categories has a
meaning and certain operations are possible. If the categorical is unordered, <code>.min()/.max()</code> will raise a <code>TypeError</code>.</p>

In [326]:
s = pd.Series(pd.Categorical(["a", "b", "c", "a"], ordered=False))

In [327]:
s = s.sort_values()

In [328]:
s = pd.Series(["a", "b", "c", "a"]).astype(CategoricalDtype(ordered=True))

In [329]:
s = s.sort_values()

In [330]:
s

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a' < 'b' < 'c']

In [331]:
s.min(), s.max()

('a', 'c')

<p>You can set categorical data to be ordered by using <code>as_ordered()</code> or unordered by using <code>as_unordered()</code>. These will by default return a <em>new</em> object.</p>

In [332]:
s.cat.as_ordered()

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a' < 'b' < 'c']

In [333]:
s.cat.as_unordered()

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a', 'b', 'c']

<p>Sorting will use the order defined by categories, not any lexical order present on the data type.
This is even true for strings and numeric data:</p>

In [334]:
s = pd.Series([1, 2, 3, 1], dtype="category")

In [335]:
s = s.cat.set_categories([2, 3, 1], ordered=True)

In [336]:
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [337]:
s = s.sort_values()

In [338]:
s

1    2
2    3
0    1
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [339]:
s.min(), s.max()

(2, 1)

## <h3>Reordering</h3>

<p>Reordering the categories is possible via the <code>Categorical.reorder_categories()</code> and the <code>Categorical.set_categories()</code> methods. For <code>Categorical.reorder_categories()</code>, all
old categories must be included in the new categories and no new categories are allowed. This will necessarily make the sort order the same as the categories order.</p>

In [340]:
s = pd.Series([1, 2, 3, 1], dtype="category")

In [341]:
s = s.cat.reorder_categories([2, 3, 1], ordered=True)

In [342]:
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [343]:
s = s.sort_values()

In [344]:
s

1    2
2    3
0    1
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [345]:
 s.min(), s.max()

(2, 1)

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Note the difference between assigning new categories and reordering the categories: the first renames categories and therefore the individual values in the <code>Series</code>, but if the first position was sorted last, the renamed value will still be sorted last. Reordering means that the way values are sorted is different afterwards, but not that individual values in the <code>Series</code> are changed.</p>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>If the <code>Categorical</code> is not ordered, <a href="../reference/api/pandas.Series.min.html#pandas.Series.min" title="pandas.Series.min"><code>Series.min()</code></a> and <a href="../reference/api/pandas.Series.max.html#pandas.Series.max" title="pandas.Series.max"><code>Series.max()</code></a> will raise <code>TypeError</code>. Numeric operations like <code>+</code>, <code>-</code>, <code>*</code>, <code>/</code> and operations based on them (e.g. <a href="../reference/api/pandas.Series.median.html#pandas.Series.median" title="pandas.Series.median"><code>Series.median()</code></a>, which would need to compute the mean between two values if the length of an array is even) do not work and raise a <code>TypeError</code>.</p>
</div>

## <h3>Multi column sorting</h3>

<p>A categorical dtyped column will participate in a multi-column sort in a similar manner to other columns.
The ordering of the categorical is determined by the <code>categories</code> of that column.</p>

In [346]:
dfs = pd.DataFrame(
    {
        "A": pd.Categorical(
            list("bbeebbaa"),
            categories=["e", "a", "b"],
            ordered=True,
        ),
        "B": [1, 2, 1, 2, 2, 1, 2, 1],
    }
)

In [347]:
dfs.sort_values(by=["A", "B"])

,A,B
2,e,1
3,e,2
7,a,1
6,a,2
0,b,1
5,b,1
1,b,2
4,b,2


<p>Reordering the <code>categories</code> changes a future sort.</p>

In [348]:
dfs["A"] = dfs["A"].cat.reorder_categories(["a", "b", "e"])

In [349]:
dfs.sort_values(by=["A", "B"])

,A,B
7,a,1
6,a,2
0,b,1
5,b,1
1,b,2
4,b,2
2,e,1
3,e,2


# <h2>Comparisons</h2>

<p>Comparing categorical data with other objects is possible in three cases:</p>

<ul>
<li><p>Comparing equality (<code>==</code> and <code>!=</code>) to a list-like object (list, Series, array, …) of the same length as the categorical data.</p></li>
<li><p>All comparisons (<code>==</code>, <code>!=</code>, <code>&gt;</code>, <code>&gt;=</code>, <code>&lt;</code>, and <code>&lt;=</code>) of categorical data to another categorical Series, when <code>ordered==True</code> and the <code>categories</code> are the same.</p></li>
<li><p>All comparisons of a categorical data to a scalar.</p></li>
</ul>

<p>All other comparisons, especially “non-equality” comparisons of two categoricals with different categories or a categorical with any list-like object, will raise a <code>TypeError</code>.</p>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>Any “non-equality” comparisons of categorical data with a <code>Series</code>, <code>np.array</code>, <code>list</code> or categorical data with different categories or ordering will raise a <code>TypeError</code> because custom categories ordering could be interpreted in two ways: one with taking into account the ordering and one without.</p>
</div>

In [350]:
cat = pd.Series([1, 2, 3]).astype(CategoricalDtype([3, 2, 1], ordered=True))

In [351]:
cat_base = pd.Series([2, 2, 2]).astype(CategoricalDtype([3, 2, 1], ordered=True))

In [352]:
cat_base2 = pd.Series([2, 2, 2]).astype(CategoricalDtype(ordered=True))

In [353]:
cat

0    1
1    2
2    3
dtype: category
Categories (3, int64): [3 < 2 < 1]

In [354]:
cat_base

0    2
1    2
2    2
dtype: category
Categories (3, int64): [3 < 2 < 1]

In [355]:
cat_base2

0    2
1    2
2    2
dtype: category
Categories (1, int64): [2]

<p>Comparing to a categorical with the same categories and ordering or to a scalar works:</p>

In [356]:
cat > cat_base

0     True
1    False
2    False
dtype: bool

In [357]:
cat > 2

0     True
1    False
2    False
dtype: bool

<p>Equality comparisons work with any list-like object of same length and scalars:</p>

In [358]:
cat == cat_base

0    False
1     True
2    False
dtype: bool

In [359]:
cat == np.array([1, 2, 3])

0    True
1    True
2    True
dtype: bool

In [360]:
cat == 2

0    False
1     True
2    False
dtype: bool

<p>This doesn’t work because the categories are not the same:</p>

In [361]:
try:
    cat > cat_base2
except TypeError as e:
    print("TypeError:", str(e))

TypeError: Categoricals can only be compared if 'categories' are the same.


<p>If you want to do a “non-equality” comparison of a categorical series with a list-like object which is not categorical data, you need to be explicit and convert the categorical data back to the original values:</p>

In [362]:
base = np.array([1, 2, 3])

In [363]:
try:
    cat > base
except TypeError as e:
    print("TypeError:", str(e))

TypeError: Cannot compare a Categorical for op __gt__ with type <class 'numpy.ndarray'>.
If you want to compare values, use 'np.asarray(cat) <op> other'.


In [364]:
np.asarray(cat) > base

array([False, False, False])

<p>When you compare two unordered categoricals with the same categories, the order is not considered:</p>

In [365]:
c1 = pd.Categorical(["a", "b"], categories=["a", "b"], ordered=False)

In [366]:
c2 = pd.Categorical(["a", "b"], categories=["b", "a"], ordered=False)

In [367]:
c1 == c2

array([ True,  True])

# <h2>Operations</h2>

<p>Apart from <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.min.html" title="pandas.Series.min"><code>Series.min()</code></a>, <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.max.html" title="pandas.Series.max"><code>Series.max()</code></a> and <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.mode.html" title="pandas.Series.mode"><code>Series.mode()</code></a>, the following operations are possible with categorical data:</p>

<p><code>Series</code> methods like <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html" title="pandas.Series.value_counts"><code>Series.value_counts()</code></a> will use all categories, even if some categories are not present in the data:</p>

In [368]:
s = pd.Series(pd.Categorical(["a", "b", "c", "c"], categories=["c", "a", "b", "d"]))

In [369]:
s.value_counts()

c    2
a    1
b    1
d    0
Name: count, dtype: int64

<p><code>DataFrame</code> methods like <a href="../reference/api/pandas.DataFrame.sum.html#pandas.DataFrame.sum" title="pandas.DataFrame.sum"><code>DataFrame.sum()</code></a> also show “unused” categories when <code>observed=False</code>.</p>

In [370]:
columns = pd.Categorical(
    ["One", "One", "Two"], categories=["One", "Two", "Three"], ordered=True
)

In [371]:
df = pd.DataFrame(
        data=[[1, 2, 3], [4, 5, 6]],
        columns=pd.MultiIndex.from_arrays([["A", "B", "B"], columns]),
).T

In [372]:
df.groupby(level=1, observed=False).sum()

,0,1
One,3,9
Two,3,6
Three,0,0


<p>Groupby will also show “unused” categories when <code>observed=False</code>:</p>

In [373]:
cats = pd.Categorical(
    ["a", "b", "b", "b", "c", "c", "c"], categories=["a", "b", "c", "d"]
)

In [374]:
df = pd.DataFrame({"cats": cats, "values": [1, 2, 2, 2, 3, 4, 5]})

In [375]:
df.groupby("cats", observed=False).mean()

,values
cats,
a,1.0
b,2.0
c,4.0
d,NaN


In [376]:
cats2 = pd.Categorical(["a", "a", "b", "b"], categories=["a", "b", "c"])

In [377]:
df2 = pd.DataFrame(
    {
        "cats": cats2,
        "B": ["c", "d", "c", "d"],
        "values": [1, 2, 3, 4],
    }
)

In [378]:
df2.groupby(["cats", "B"], observed=False).mean()

values
cats B        
a    c     1.0
     d     2.0
b    c     3.0
     d     4.0
c    c     NaN
     d     NaN

<p>Pivot tables:</p>

In [379]:
raw_cat = pd.Categorical(["a", "a", "b", "b"], categories=["a", "b", "c"])

In [380]:
df = pd.DataFrame({"A": raw_cat, "B": ["c", "d", "c", "d"], "values": [1, 2, 3, 4]})

In [381]:
pd.pivot_table(df, values="values", index=["A", "B"], observed=False)

values
A B        
a c     1.0
  d     2.0
b c     3.0
  d     4.0

# <h2>Data munging</h2>

<p>The optimized pandas data access methods  <code>.loc</code>, <code>.iloc</code>, <code>.at</code>, and <code>.iat</code>, work as normal. The only difference is the return type (for getting) and that only values already in <code>categories</code> can be assigned.</p>

## <h3>Getting</h3>

<p>If the slicing operation returns either a <code>DataFrame</code> or a column of type <code>Series</code>, the <code>category</code> dtype is preserved.</p>

In [382]:
idx = pd.Index(["h", "i", "j", "k", "l", "m", "n"])

In [383]:
cats = pd.Series(["a", "b", "b", "b", "c", "c", "c"], dtype="category", index=idx)

In [384]:
values = [1, 2, 2, 2, 3, 4, 5]

In [385]:
df = pd.DataFrame({"cats": cats, "values": values}, index=idx)

In [386]:
df.iloc[2:4, :]

,cats,values
j,b,2
k,b,2


In [387]:
df.iloc[2:4, :].dtypes

cats      category
values       int64
dtype: object

In [388]:
df.loc["h":"j", "cats"]

h    a
i    b
j    b
Name: cats, dtype: category
Categories (3, object): ['a', 'b', 'c']

In [389]:
df[df["cats"] == "b"]

,cats,values
i,b,2
j,b,2
k,b,2


<p>An example where the category type is not preserved is if you take one single row: the resulting <code>Series</code> is of dtype <code>object</code>:</p>

In [390]:
# get the complete "h" row as a Series
df.loc["h", :]

cats      a
values    1
Name: h, dtype: object

<p>Returning a single item from categorical data will also return the value, not a categorical of length “1”.</p>

In [391]:
df.iat[0, 0]

'a'

In [392]:
df["cats"] = df["cats"].cat.rename_categories(["x", "y", "z"])

In [393]:
df.at["h", "cats"]  # returns a string

'x'

<div class="alert alert-block alert-info">
<p>Note</p>
<p>This is in contrast to R’s <code>factor</code> function, where <code>factor(c(1,2,3))[1]</code> returns a single value <code>factor</code>.</p>
</div>

<p>To get a single value <code>Series</code> of type <code>category</code>, you pass in a list with
a single value:</p>

In [394]:
df.loc[["h"], "cats"]

h    x
Name: cats, dtype: category
Categories (3, object): ['x', 'y', 'z']

## <h3>String and datetime accessors</h3>

<p>The accessors  <code>.dt</code> and <code>.str</code> will work if the <code>s.cat.categories</code> are of
an appropriate type:</p>

In [395]:
str_s = pd.Series(list("aabb"))

In [396]:
str_cat = str_s.astype("category")

In [397]:
str_cat

0    a
1    a
2    b
3    b
dtype: category
Categories (2, object): ['a', 'b']

In [398]:
str_cat.str.contains("a")

0     True
1     True
2    False
3    False
dtype: bool

In [399]:
date_s = pd.Series(pd.date_range("1/1/2015", periods=5))

In [400]:
date_cat = date_s.astype("category")

In [401]:
date_cat

0   2015-01-01
1   2015-01-02
2   2015-01-03
3   2015-01-04
4   2015-01-05
dtype: category
Categories (5, datetime64[ns]): [2015-01-01, 2015-01-02, 2015-01-03, 2015-01-04, 2015-01-05]

In [402]:
date_cat.dt.day

0    1
1    2
2    3
3    4
4    5
dtype: int32

<div class="alert alert-block alert-info">
<p>Note</p>
<p>The returned <code>Series</code> (or <code>DataFrame</code>) is of the same type as if you used the <code>.str.&lt;method&gt;</code> / <code>.dt.&lt;method&gt;</code> on a <code>Series</code> of that type (and not of type <code>category</code>!).</p>
</div>

<p>That means, that the returned values from methods and properties on the accessors of a <code>Series</code> and the returned values from methods and properties on the accessors of this <code>Series</code> transformed to one of type <code>category</code> will be equal:</p>

In [403]:
ret_s = str_s.str.contains("a")

In [404]:
ret_cat = str_cat.str.contains("a")

In [405]:
ret_s.dtype == ret_cat.dtype

True

In [406]:
ret_s == ret_cat

0    True
1    True
2    True
3    True
dtype: bool

<div class="alert alert-block alert-info">
<p>Note</p>
<p>The work is done on the <code>categories</code> and then a new <code>Series</code> is constructed. This has
some performance implication if you have a <code>Series</code> of type string, where lots of elements
are repeated (i.e. the number of unique elements in the <code>Series</code> is a lot smaller than the
length of the <code>Series</code>). In this case it can be faster to convert the original <code>Series</code>
to one of type <code>category</code> and use <code>.str.&lt;method&gt;</code> or <code>.dt.&lt;property&gt;</code> on that.</p>
</div>

## <h3>Setting</h3>

<p>Setting values in a categorical column (or <code>Series</code>) works as long as the value is included in the <code>categories</code>:</p>

In [407]:
idx = pd.Index(["h", "i", "j", "k", "l", "m", "n"])

In [408]:
cats = pd.Categorical(["a", "a", "a", "a", "a", "a", "a"], categories=["a", "b"])

In [409]:
values = [1, 1, 1, 1, 1, 1, 1]

In [410]:
df = pd.DataFrame({"cats": cats, "values": values}, index=idx)

In [411]:
df.iloc[2:4, :] = [["b", 2], ["b", 2]]

In [412]:
df

,cats,values
h,a,1
i,a,1
j,b,2
k,b,2
l,a,1
m,a,1
n,a,1


In [413]:
try:
    df.iloc[2:4, :] = [["c", 3], ["c", 3]]
except TypeError as e:
    print("TypeError:", str(e))

TypeError: Cannot setitem on a Categorical with a new category, set the categories first


<p>Setting values by assigning categorical data will also check that the <code>categories</code> match:</p>

In [414]:
df.loc["j":"k", "cats"] = pd.Categorical(["a", "a"], categories=["a", "b"])

In [415]:
df

,cats,values
h,a,1
i,a,1
j,a,2
k,a,2
l,a,1
m,a,1
n,a,1


In [416]:
try:
    df.loc["j":"k", "cats"] = pd.Categorical(["b", "b"], categories=["a", "b", "c"])
except TypeError as e:
    print("TypeError:", str(e))

TypeError: Cannot set a Categorical with another, without identical categories


<p>Assigning a <code>Categorical</code> to parts of a column of other types will use the values:</p>

In [417]:
df = pd.DataFrame({"a": [1, 1, 1, 1, 1], "b": ["a", "a", "a", "a", "a"]})

df.loc[1:2, "a"] = pd.Categorical(["b", "b"], categories=["a", "b"])

df.loc[2:3, "b"] = pd.Categorical(["b", "b"], categories=["a", "b"])

/var/folders/rt/k30tjkvx04vdq1qxvtwsb47m0000gn/T/ipykernel_57101/2732893942.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['b', 'b']
Categories (2, object): ['a', 'b']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[1:2, "a"] = pd.Categorical(["b", "b"], categories=["a", "b"])


In [418]:
df

,a,b
0,1,a
1,b,a
2,b,b
3,1,b
4,1,a


In [419]:
df.dtypes

a    object
b    object
dtype: object

## <h3>Merging / concatenation</h3>

<p>By default, combining <code>Series</code> or <code>DataFrames</code> which contain the same categories results in <code>category</code> dtype, otherwise results will depend on the dtype of the underlying categories. Merges that result in non-categorical dtypes will likely have higher memory usage. Use <code>.astype</code> or <code>union_categoricals</code> to ensure <code>category</code> results.</p>

In [420]:
from pandas.api.types import union_categoricals

In [421]:
# same categories
s1 = pd.Series(["a", "b"], dtype="category")

In [422]:
s2 = pd.Series(["a", "b", "a"], dtype="category")

In [423]:
pd.concat([s1, s2])

0    a
1    b
0    a
1    b
2    a
dtype: category
Categories (2, object): ['a', 'b']

In [424]:
# different categories
s3 = pd.Series(["b", "c"], dtype="category")

In [425]:
pd.concat([s1, s3])

0    a
1    b
0    b
1    c
dtype: object

In [426]:
# Output dtype is inferred based on categories values
int_cats = pd.Series([1, 2], dtype="category")

In [427]:
float_cats = pd.Series([3.0, 4.0], dtype="category")

In [428]:
pd.concat([int_cats, float_cats])

0    1.0
1    2.0
0    3.0
1    4.0
dtype: float64

In [429]:
pd.concat([s1, s3]).astype("category")

0    a
1    b
0    b
1    c
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [430]:
union_categoricals([s1.array, s3.array])

['a', 'b', 'b', 'c']
Categories (3, object): ['a', 'b', 'c']

<p>The following table summarizes the results of merging <code>Categoricals</code>:</p>
<table class="table">
<thead>
<tr class="row-odd"><th class="head"><p>arg1</p></th>
<th class="head"><p>arg2</p></th>
<th class="head"><p>identical</p></th>
<th class="head"><p>result</p></th>
</tr>
</thead>
<tbody>
<tr class="row-even"><td><p>category</p></td>
<td><p>category</p></td>
<td><p>True</p></td>
<td><p>category</p></td>
</tr>
<tr class="row-odd"><td><p>category (object)</p></td>
<td><p>category (object)</p></td>
<td><p>False</p></td>
<td><p>object (dtype is inferred)</p></td>
</tr>
<tr class="row-even"><td><p>category (int)</p></td>
<td><p>category (float)</p></td>
<td><p>False</p></td>
<td><p>float (dtype is inferred)</p></td>
</tr>
</tbody>
</table>

## <h3>Unioning</h3>

<p>If you want to combine categoricals that do not necessarily have the same categories, the <a href="../reference/api/pandas.api.types.union_categoricals.html#pandas.api.types.union_categoricals" title="pandas.api.types.union_categoricals"><code>union_categoricals()</code></a> function will combine a list-like of categoricals.The new categories will be the union of the categories being combined.</p>

In [431]:
from pandas.api.types import union_categoricals

In [432]:
a = pd.Categorical(["b", "c"])

In [433]:
b = pd.Categorical(["a", "b"])

In [434]:
union_categoricals([a, b])

['b', 'c', 'a', 'b']
Categories (3, object): ['b', 'c', 'a']

<p>By default, the resulting categories will be ordered as they appear in the data. If you want the categories to be lexsorted, use <code>sort_categories=True</code> argument.</p>

In [435]:
union_categoricals([a, b], sort_categories=True)

['b', 'c', 'a', 'b']
Categories (3, object): ['a', 'b', 'c']

<p><code>union_categoricals</code> also works with the “easy” case of combining two categoricals of the same categories and order information (e.g. what you could also <code>append</code> for).</p>

In [436]:
a = pd.Categorical(["a", "b"], ordered=True)

In [437]:
b = pd.Categorical(["a", "b", "a"], ordered=True)

In [438]:
union_categoricals([a, b])

['a', 'b', 'a', 'b', 'a']
Categories (2, object): ['a' < 'b']

<p>The below raises <code>TypeError</code> because the categories are ordered and not identical.</p>

In [439]:
a = pd.Categorical(["a", "b"], ordered=True)

In [440]:
b = pd.Categorical(["a", "b", "c"], ordered=True)

In [441]:
union_categoricals([a, b])

TypeError: to union ordered Categoricals, all categories must be the same

<p>Ordered categoricals with different categories or orderings can be combined by using the <code>ignore_ordered=True</code> argument.</p>

In [444]:
a = pd.Categorical(["a", "b", "c"], ordered=True)

In [445]:
b = pd.Categorical(["c", "b", "a"], ordered=True)

In [446]:
union_categoricals([a, b], ignore_order=True)

['a', 'b', 'c', 'c', 'b', 'a']
Categories (3, object): ['a', 'b', 'c']

<p><a href="https://pandas.pydata.org/docs/reference/api/pandas.api.types.union_categoricals.html" title="pandas.api.types.union_categoricals"><code>union_categoricals()</code></a> also works with a <code>CategoricalIndex</code>, or <code>Series</code> containing categorical data, but note that the resulting array will always be a plain <code>Categorical</code>:</p>

In [ ]:
a = pd.Series(["b", "c"], dtype="category")

In [448]:
b = pd.Series(["a", "b"], dtype="category")

In [450]:
union_categoricals([a, b])

['b', 'c', 'a', 'b']
Categories (3, object): ['b', 'c', 'a']

<div class="alert alert-block alert-info">
<p>Note</p>
<p><code>union_categoricals</code> may recode the integer codes for categories
when combining categoricals.  This is likely what you want,
but if you are relying on the exact numbering of the categories, be
aware.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre><span>In [212]: c1 = pd.Categorical(["b", "c"])

In [213]: c2 = pd.Categorical(["a", "b"])

In [214]: c1
<span class="gh">Out[214]:
<span class="go">['b', 'c']
<span class="go">Categories (2, object): ['b', 'c']

<span class="go"># "b" is coded to 0
In [215]: c1.codes
<span class="gh">Out[215]: <span class="go">array([0, 1], dtype=int8)

In [216]: c2
<span class="gh">Out[216]:
<span class="go">['a', 'b']
<span class="go">Categories (2, object): ['a', 'b']

<span class="go"># "b" is coded to 1
In [217]: c2.codes
<span class="gh">Out[217]: <span class="go">array([0, 1], dtype=int8)

In [218]: c = union_categoricals([c1, c2])

In [219]: c
<span class="gh">Out[219]:
<span class="go">['b', 'c', 'a', 'b']
<span class="go">Categories (3, object): ['b', 'c', 'a']

<span class="go"># "b" is coded to 0 throughout, same as c1, different from c2
In [220]: c.codes
<span class="gh">Out[220]: <span class="go">array([0, 1, 2, 0], dtype=int8)
</pre>
</div>
</div>
</div>

# <h2>Getting data in/out</h2>

<p>You can write data that contains <code>category</code> dtypes to a <code>HDFStore</code>. See <a href="https://pandas.pydata.org/docs/user_guide/io.html#io-hdf5-categorical">here</a> for an example and caveats.</p>

<p>It is also possible to write data to and reading data from <em>Stata</em> format files. See <a href="https://pandas.pydata.org/docs/user_guide/io.html#io-stata-categorical">here</a> for an example and caveats.</p>

<p>Writing to a CSV file will convert the data, effectively removing any information about the categorical (categories and ordering). So if you read back the CSV file you have to convert the relevant columns back to <code>category</code> and assign the right categories and categories ordering.</p>

In [451]:
import io

In [452]:
s = pd.Series(pd.Categorical(["a", "b", "b", "a", "a", "d"]))

In [453]:
# rename the categories
s = s.cat.rename_categories(["very good", "good", "bad"])

In [454]:
# reorder the categories and add missing categories
s = s.cat.set_categories(["very bad", "bad", "medium", "good", "very good"])

In [455]:
df = pd.DataFrame({"cats": s, "vals": [1, 2, 3, 4, 5, 6]})

In [456]:
csv = io.StringIO()

In [457]:
df.to_csv(csv)

In [463]:
df2.dtypes

Unnamed: 0     int64
cats          object
vals           int64
dtype: object

In [458]:
df2 = pd.read_csv(io.StringIO(csv.getvalue()))

In [464]:
df2["cats"]

0    very good
1         good
2         good
3    very good
4    very good
5          bad
Name: cats, dtype: object

In [465]:
# Redo the category
df2["cats"] = df2["cats"].astype("category")

In [466]:
df2["cats"] = df2["cats"].cat.set_categories(
    ["very bad", "bad", "medium", "good", "very good"]
)

In [467]:
df2.dtypes

Unnamed: 0       int64
cats          category
vals             int64
dtype: object

In [468]:
df2["cats"]

0    very good
1         good
2         good
3    very good
4    very good
5          bad
Name: cats, dtype: category
Categories (5, object): ['very bad', 'bad', 'medium', 'good', 'very good']

<p>The same holds for writing to a SQL database with <code>to_sql</code>.</p>

# <h2>Missing data</h2>

<p>pandas primarily uses the value <code>np.nan</code> to represent missing data. It is by default not included in computations. See the <a href="https://pandas.pydata.org/docs/user_guide/missing_data.html">Missing Data section</a>.</p>

<p>Missing values should <strong>not</strong> be included in the Categorical’s <code>categories</code>, only in the <code>values</code>.
Instead, it is understood that NaN is different, and is always a possibility. When working with the Categorical’s <code>codes</code>, missing values will always have a code of <code>-1</code>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell61"><span>In [235]: s = pd.Series(["a", "b", np.nan, "a"], dtype="category")

<span class="go"># only two categories
In [236]: s
<span class="gh">Out[236]:
<span class="go">0      a
<span class="go">1      b
<span class="go">2    NaN
<span class="go">3      a
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']

In [237]: s.cat.codes
<span class="gh">Out[237]:
<span class="go">0    0
<span class="go">1    1
<span class="go">2   -1
<span class="go">3    0
<span class="go">dtype: int8
</pre>
</div>
</div>

<p>Methods for working with missing data, e.g. <a href="../reference/api/pandas.Series.isna.html#pandas.Series.isna" title="pandas.Series.isna"><code>isna()</a>, <a href="../reference/api/pandas.Series.fillna.html#pandas.Series.fillna" title="pandas.Series.fillna"><code>fillna()</code></a>, <a href="../reference/api/pandas.Series.dropna.html#pandas.Series.dropna" title="pandas.Series.dropna"><code>dropna()</code></a>, all work normally:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell62"><span>lass="gp">In [238]: s = pd.Series(["a", "b", np.nan], dtype="category")

In [239]: s
<span class="gh">Out[239]:
<span class="go">0      a
<span class="go">1      b
<span class="go">2    NaN
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']

In [240]: pd.isna(s)
<span class="gh">Out[240]:
<span class="go">0    False
<span class="go">1    False
<span class="go">2     True
<span class="go">dtype: bool

In [241]: s.fillna("a")
<span class="gh">Out[241]:
<span class="go">0    a
<span class="go">1    b
<span class="go">2    a
<span class="go">dtype: category
<span class="go">Categories (2, object): ['a', 'b']
</pre>
</div>
</div>

# <h2>Differences to R’s <code>factor</h2>

<p>The following differences to R’s factor functions can be observed:</p>

<ul>

<li><p>R’s <code>levels</code> are named <code>categories</code>.</p></li>

<li><p>R’s <code>levels</code> are always of type string, while <code>categories</code> in pandas can be of any dtype.</p></li>

<li><p>It’s not possible to specify labels at creation time. Use <code>s.cat.rename_categories(new_labels)</code>
afterwards.</p></li>

<li><p>In contrast to R’s <code>factor</code> function, using categorical data as the sole input to create a new categorical series will <em>not</em> remove unused categories but create a new categorical series which is equal to the passed in one!</p></li>

<li><p>R allows for missing values to be included in its <code>levels</code> (pandas’ <code>categories</code>). Pandas
does not allow <code>NaN</code> categories, but missing values can still be in the <code>values</code>.</p></li>

</ul>

# <h2>Gotchas</h2>

## <h3>Memory usage</h3>

<p>The memory usage of a <code>Categorical is proportional to the number of categories plus the length of the data. In contrast, an <code>object</code> dtype is a constant times the length of the data.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell63"><span>In [242]: s = pd.Series(["foo", "bar"] * 1000)

<span class="go"># object dtype
In [243]: s.nbytes
<span class="gh">Out[243]: <span class="go">16000

<span class="go"># category dtype
In [244]: s.astype("category").nbytes
<span class="gh">Out[244]: <span class="go">2016
</pre>
</div>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>If the number of categories approaches the length of the data, the <code>Categorical</code> will use nearly the same ormore memory than an equivalent <code>object</code> dtype representation.</p>
    
<div class="highlight-ipython notranslate"><div class="highlight"><pre><span>In [245]: s = pd.Series(["foo%04d" % i for i in range(2000)])

<span class="go"># object dtype
In [246]: s.nbytes
<span class="gh">Out[246]: <span class="go">16000

<span class="go"># category dtype
In [247]: s.astype("category").nbytes
<span class="gh">Out[247]: <span class="go">20000
</pre>
</div>
</div>
</div>

## <h3><code>Categorical</code> is not a <code>numpy</code> array</h3>

<p>Currently, categorical data and the underlying <code>Categorical</code> is implemented as a Python object and not as a low-level NumPy array dtype. This leads to some problems.</p>

<p>NumPy itself doesn’t know about the new <code>dtype</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre><span>In [248]: try:
    np.dtype("category")
except TypeError as e:
    print("TypeError:", str(e))

<span class="go">TypeError: data type 'category' not understood

In [249]: dtype = pd.Categorical(["a"]).dtype

In [250]: try:
    np.dtype(dtype)
except TypeError as e:
    print("TypeError:", str(e))

<span class="go">TypeError: Cannot interpret 'CategoricalDtype(categories=['a'], ordered=False, categories_dtype=object)' as a data type
</pre>
</div>
</div>

<p>Dtype comparisons work:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell66"><span>In [251]: dtype == np.str_
<span class="gh">Out[251]: <span class="go">False

In [252]: np.str_ == dtype
<span class="gh">Out[252]: <span class="go">False
</pre>
</div>
</div>

<p>To check if a Series contains Categorical data, use <code>hasattr(s, 'cat')</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell67"><span>In [253]: hasattr(pd.Series(["a"], dtype="category"), "cat")
<span class="gh">Out[253]: <span class="go">True

In [254]: hasattr(pd.Series(["a"]), "cat")
<span class="gh">Out[254]: <span class="go">False
</pre>
</div>
</div>

<p>Using NumPy functions on a <code>Series</code> of type <code>category</code> should not work as <code>Categoricals</code> are not numeric data (even in the case that <code>.categories</code> is numeric).</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell68"><span>In [255]: s = pd.Series(pd.Categorical([1, 2, 3, 4]))

In [256]: try:
    np.sum(s)
except TypeError as e:
    print("TypeError:", str(e))

<span class="go">TypeError: 'Categorical' with dtype category does not support reduction 'sum'
</pre>
</div>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>If such a function works, please file a bug at <a class="github reference external" href="https://github.com/pandas-dev/pandas">pandas-dev/pandas</a>!</p>
</div>

## <h3>dtype in apply</h3>

<p>pandas currently does not preserve the dtype in apply functions: If you apply along rows you get a <code>Series</code> of <code>object</code> <code>dtype</code> (same as getting a row -&gt; getting one element will return a basic type) and applying along columns will also convert to object. <code>NaN</code> values are unaffected.
You can use <code>fillna</code> to handle missing values before applying a function.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell69"><span>In [257]: df = pd.DataFrame(
    {
        "a": [1, 2, 3, 4],
        "b": ["a", "b", "c", "d"],
        "cats": pd.Categorical([1, 2, 3, 2]),
    }
)


In [258]: df.apply(lambda row: type(row["cats"]), axis=1)
<span class="gh">Out[258]:
<span class="go">0    &lt;class 'int'&gt;
<span class="go">1    &lt;class 'int'&gt;
<span class="go">2    &lt;class 'int'&gt;
<span class="go">3    &lt;class 'int'&gt;
<span class="go">dtype: object

In [259]: df.apply(lambda col: col.dtype, axis=0)
<span class="gh">Out[259]:
<span class="go">a          int64
<span class="go">b         object
<span class="go">cats    category
<span class="go">dtype: object
</pre>
</div>
</div>

## <h3>Categorical index</h3>

<p><code>CategoricalIndex</code> is a type of index that is useful for supporting indexing with duplicates. This is a container around a <code>Categorical</code> and allows efficient indexing and storage of an index with a large number of duplicated elements.
See the <a href="advanced.html#advanced-categoricalindex">advanced indexing docs</a> for a more detailed explanation.</p>

<p>Setting the index will create a <code>CategoricalIndex</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell70"><span>In [260]: cats = pd.Categorical([1, 2, 3, 4], categories=[4, 2, 3, 1])

In [261]: strings = ["a", "b", "c", "d"]

In [262]: values = [4, 2, 3, 1]

In [263]: df = pd.DataFrame({"strings": strings, "values": values}, index=cats)

In [264]: df.index
<span class="gh">Out[264]: <span class="go">CategoricalIndex([1, 2, 3, 4], categories=[4, 2, 3, 1], ordered=False, dtype='category')

<span class="go"># This now sorts by the categories order
In [265]: df.sort_index()
<span class="gh">Out[265]:
<span class="go">  strings  values
<span class="go">4       d       1
<span class="go">2       b       2
<span class="go">3       c       3
<span class="go">1       a       4
</pre>
</div>
</div>

## <h3>Side effects</h3>

<p>Constructing a <code>Series</code> from a <code>Categorical</code> will not copy the input <code>Categorical</code>. This means that changes to the <code>Series</code> will in most cases change the original <code>Categorical</code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell71"><span>In [266]: cat = pd.Categorical([1, 2, 3, 10], categories=[1, 2, 3, 4, 10])

In [267]: s = pd.Series(cat, name="cat")

In [268]: cat
<span class="gh">Out[268]:
<span class="go">[1, 2, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]

In [269]: s.iloc[0:2] = 10

In [270]: cat
<span class="gh">Out[270]:
<span class="go">[10, 10, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]
</pre>
</div>
</div>

<p>Use <code>copy=True</code> to prevent such a behaviour or simply don’t reuse <code>Categoricals</code>:</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell72"><span>In [271]: cat = pd.Categorical([1, 2, 3, 10], categories=[1, 2, 3, 4, 10])

In [272]: s = pd.Series(cat, name="cat", copy=<span class="kc">True)

In [273]: cat
<span class="gh">Out[273]:
<span class="go">[1, 2, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]

In [274]: s.iloc[0:2] = 10

In [275]: cat
<span class="gh">Out[275]:
<span class="go">[1, 2, 3, 10]
<span class="go">Categories (5, int64): [1, 2, 3, 4, 10]
</pre>
</div>
</div>

<div class="alert alert-block alert-info">
<p>Note</p>
<p>This also happens in some cases when you supply a NumPy array instead of a <code>Categorical</code>:
using an int array (e.g. <code>np.array([1,2,3,4])</code>) will exhibit the same behavior, while using
a string array (e.g. <code>np.array(["a","b","c","a"])</code>) will not.</p>
</div>